# Precision weed spraying: run everything on Google Colab

**Runtime → Change runtime type → T4 GPU**, then run the cells in order (about 25 to 35 minutes in total).

In [ ]:
!git clone https://github.com/GHasnain14/precision-weed-spraying.git
%cd precision-weed-spraying
!pip install -q gdown

## 1. Download DeepWeeds (468 MB of images, CC BY 4.0)

In [ ]:
!python src/data.py --root data

## 2. Train MobileNetV3 on official fold 0

In [ ]:
!python src/train.py --root data --arch mobilenet_v3_large --epochs 12 --workers 2

## 3. Predict on validation and test sets

In [ ]:
!python src/evaluate.py --root data --ckpt checkpoints/mobilenet_v3_large_fold0.pt

## 4. Measure inference time (limits sprayer speed)

In [ ]:
!python src/latency.py --arch mobilenet_v3_large --cpu-threads 2

## 5. Spray decisions: threshold chosen on validation, reported on test

In [ ]:
!python src/spray.py --results results --footprint 0.45

In [ ]:
!python src/make_figures.py --results results
from IPython.display import Image, display
display(Image('results/spray_tradeoff.png')); display(Image('results/confusion_matrix.png'))

## 6. Download the results folder and send it back

In [ ]:
!zip -qr results.zip results checkpoints/*_history.json
from google.colab import files
files.download('results.zip')